# R3 — Restaurant SCD Type 2

Builds `hop.silver.dim_restaurant_scd2`, the full restaurant history from the out-of-order CDC feed, using Auto CDC sequenced on `op_ts`. The transformation code runs in the **hop_pipeline** SDP (`src/hop_pipeline/transformations/dim_restaurant_scd2.py`); this notebook documents the round and verifies its outputs.

In [0]:
# Config cell — change model names here only, if a built-in model name changes
CATALOG = "hop"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"
print(CATALOG, CHAT_MODEL, EMBED_MODEL)

## What R3 builds (in the hop_pipeline SDP)

* `hop.silver.dim_restaurant_scd2` — SCD Type 2 keyed on `restaurant_id`, sequenced by `op_ts`, with `__START_AT` / `__END_AT` (open version has an empty `__END_AT`)
* One row per restaurant version; late-arriving CDC rows are folded into history by sequence, not arrival order

In [0]:
# R3 — verification (read-only): version counts and overlap checks
from pyspark.sql import functions as F

d = spark.table("hop.silver.dim_restaurant_scd2")
print("versions:", d.count(), "| restaurants:", d.select("restaurant_id").distinct().count(),
      "| open versions:", d.filter("__END_AT IS NULL").count())
print("duplicate (restaurant_id, __START_AT):",
      d.groupBy("restaurant_id", "__START_AT").count().filter("count > 1").count())

a = d.alias("a").join(d.alias("b"), "restaurant_id").filter(
    (F.col("a.__START_AT") < F.col("b.__START_AT"))
    & (F.col("b.__START_AT") < F.col("a.__END_AT"))
    & F.col("a.__END_AT").isNotNull() & F.col("b.__END_AT").isNotNull())
print("overlapping closed versions:", a.count())
display(d.filter("restaurant_id = 'RST-066'").orderBy("__START_AT"))